In [1]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

In [2]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'
gen_per_res_vs_marg_cost = '../../data/XM-API/Plan/gen_per_res_vs_marg_cost/Esc0.csv'
emissions = '../../data/XM-API/Plan/emissions/esc0.csv'
cap_2023 = '../../data/XM-API/Plan/installed_capacity/esc0/2023.csv'
cap_2037 = '../../data/XM-API/Plan/installed_capacity/esc0/2037.csv'
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

In [3]:
gen_info = pd.read_csv(model_inputs_path+'gen_info.csv')
gen_build_predetermined = pd.read_csv(model_inputs_path+'gen_build_predetermined.csv')

df = pd.merge(gen_info, gen_build_predetermined, on='GENERATION_PROJECT')
# Select only desired values
df = df[['GENERATION_PROJECT', 'gen_tech', 'build_year', 'build_gen_predetermined','gen_load_zone']]
# Change values on gen_tech to match Switch output
df['gen_tech'] = df['gen_tech'].replace(parse_tech)

# Rename columns
df.columns = ['Project', 'Technology', 'Year', 'Capacity (MW)','Zone']

df = df[df['Year'] <= 2023]
df.head()

,Project,Technology,Year,Capacity (MW),Zone
0,AMOYA,Run of River,2013,80.0,Surocciden
1,BARRANQUILL3,Thermal,1980,60.0,Caribe
2,BARRANQUILL4,Thermal,1980,60.0,Caribe
3,BETANIA,Hydro,1987,540.0,Surocciden
4,C_LLERAS_R,Run of River,2015,78.0,Antioquia


In [32]:
import plotly.express as px

fig = px.scatter(
    df, x='Year', y='Capacity (MW)',  color='Technology', color_discrete_map=tech_colors,category_orders={"Technology": tech_order},
    hover_name='Project',
    title="Installed Capacity over time",
    height=9*50, width=16*50,
    template='plotly_white',
    labels={'Capacity (MW)': 'Installed Capacity (MW)','Technology': ''}
    )
fig.add_annotation(
    x=2021,
    y=1200,
    text='Ituango Phase I',
    showarrow=True,
    arrowhead=1,
    ax=-20,  # hacia la izquierda
    ay=-20,  # hacia arriba
    font=dict(
        family="Arial",
        size=12,
        color="black"
    ),
    bgcolor="rgba(255, 255, 255, 0.7)",
)
fig.update_layout(
    legend=dict(
        orientation="h",
        y=-0.2, x=0.5,
        xanchor='center'
    ),
    font_family="Arial", font_size=14,
    yaxis=dict(range=[-100, 1300])
)

fig.write_image(f"../images/Installed Capacity over Time.png")
fig.show()

In [11]:
cap_by_zone = df.copy().groupby(['Technology','Zone']).agg({
    'Capacity (MW)': 'sum'
}).reset_index()
cap_by_zone['Zone'] = cap_by_zone['Zone'].replace({'Surocciden': 'Suroccidente'})
# Create a bar plot
fig = px.bar(
    cap_by_zone, x='Zone', y='Capacity (MW)', color='Technology', color_discrete_map=tech_colors,category_orders={"Technology": tech_order},
    barmode='stack', labels={'Capacity (MW)': 'Installed Capacity (MW)', 'Zone': 'Zone', 'Technology': ''},
    height=9*50, width=16*50,
    template='plotly_white',
    )
fig.update_layout(
    legend=dict(
        orientation="h",
        y=-0.2, x=0.5,
        xanchor='center'
    ),
    font_family="Arial", font_size=14
)

fig.write_image(f"../images/Installed Capacity by Tech.png")
# Show the plot
fig.show()

In [6]:
cap_by_tech = cap_by_zone.copy().groupby(['Technology']).agg({
    'Capacity (MW)': 'sum'
}).reset_index()
cap_by_tech['Capacity (GW)'] = cap_by_tech['Capacity (MW)']/1000
cap_total = cap_by_tech['Capacity (GW)'].sum()
cap_by_tech['Percentage'] = cap_by_tech['Capacity (GW)']*100/cap_total
print(cap_total)
cap_by_tech

19.797642


,Technology,Capacity (MW),Capacity (GW),Percentage
0,Hydro,12391.502,12.391502,62.590797
1,Run of River,782.050,0.782050,3.950218
2,Solar,752.170,0.752170,3.799291
3,Thermal,5821.500,5.821500,29.405017
4,Wind,50.420,0.050420,0.254677
